In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import os

# Caminho da pasta db
db_path = "../db"

def normalize_channels(df, ch_cols):
    """
    Normaliza cada canal (z-score) dentro de UM arquivo (um voluntário):
    valor_norm = (x - média) / desvio_padrão
    """
    df = df.copy()
    for ch in ch_cols:
        mu = df[ch].mean()
        sigma = df[ch].std()
        if sigma == 0:
            sigma = 1.0
        df[ch] = (df[ch] - mu) / sigma
    return df

def load_group(pattern):
    """
    Carrega todos os arquivos que batem com o pattern (ex: voluntary_*_center.csv),
    normaliza por arquivo e concatena.
    NÃO usa Timestamp – o índice vai ser o "tempo".
    """
    files = sorted(glob.glob(os.path.join(db_path, pattern)))
    print(f"Encontrados {len(files)} arquivos para {pattern}")
    
    df_list = []
    for f in files:
        df = pd.read_csv(f)

        # identifica colunas de canais
        ch_cols = [c for c in df.columns if c.startswith("CH_")]

        # normaliza canais por voluntário
        df = normalize_channels(df, ch_cols)

        # opcional: guardar ID do voluntário (ex: 001 de voluntary_001_center.csv)
        base = os.path.basename(f)              # voluntary_001_center.csv
        parts = base.split("_")                 # ["voluntary", "001", "center.csv"]
        subj_id = parts[1] if len(parts) > 1 else "UNK"
        df["Subject"] = subj_id

        # descartamos Timestamp – só vamos usar o índice
        if "Timestamp" in df.columns:
            df = df.drop(columns=["Timestamp"])

        df_list.append(df)
    
    return pd.concat(df_list, ignore_index=True)

# --- Carregar dados normalizados (sem Timestamp) ---
center_data = load_group("voluntary_*_center.csv")
left_data   = load_group("voluntary_*_left.csv")
right_data  = load_group("voluntary_*_right.csv")

print(center_data.head())
print(center_data.shape)

print(left_data.head())
print(left_data.shape)

print(right_data.head())
print(right_data.shape)


In [ ]:
for i in range(1, 9):
    ch = f"CH_{i}"

    plt.figure(figsize=(12, 6))

    plt.plot(center_data.index, center_data[ch], label='Center EMG', alpha=0.7)
    plt.plot(left_data.index,   left_data[ch],   label='Left EMG', alpha=0.7)
    plt.plot(right_data.index,  right_data[ch],  label='Right EMG', alpha=0.7)

    plt.xlabel('Sample Index')
    plt.ylabel(f'EMG Channel {i} (normalized)')
    plt.title(f'EMG Channel {i}: Center vs Left vs Right')
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import kurtosis, skew
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import LabelEncoder
import pickle
import os

# ----------------------------------------------
# MATH FUNCTIONS
# ----------------------------------------------

def zero_crossings(x):
    return ((x[:-1] * x[1:]) < 0).sum()

def slope_sign_changes(x):
    return np.sum((np.diff(np.sign(np.diff(x)))) != 0)

def waveform_length(x):
    return np.sum(np.abs(np.diff(x)))

# ----------------------------------------------
# SLIDING WINDOW EXTRACTOR
# ----------------------------------------------

def extract_sliding_features(df, sensor_cols, window_size=15, step_size=1):

    df['block_id'] = (df['State'] != df['State'].shift()).cumsum()

    all_features = []
    all_labels = []

    for block_id, group in df.groupby('block_id'):

        label = group['State'].iloc[0]
        data = group[sensor_cols].values
        L = len(data)

        if L < window_size:
            continue

        for i in range(0, L - window_size + 1, step_size):
            window = data[i:i+window_size]
            row = []

            # 72 features
            for ch in range(window.shape[1]):
                x = window[:, ch]

                feats = [
                    np.sqrt(np.mean(x**2)),
                    np.mean(np.abs(x)),
                    zero_crossings(x),
                    slope_sign_changes(x),
                    waveform_length(x),
                    np.var(x),
                    np.mean(x),
                    kurtosis(x) if np.std(x) > 0 else 0,
                    skew(x) if np.std(x) > 0 else 0
                ]

                row.extend(feats)

            all_features.append(row)
            all_labels.append(label)

    # DataFrame 72 features
    feat_suffixes = ['RMS','MAV','ZC','SSC','WL','VAR','MEAN','KURT','SKEW']
    col_names = [f"{ch}_{f}" for ch in sensor_cols for f in feat_suffixes]

    X_df = pd.DataFrame(all_features, columns=col_names)
    return X_df, all_labels


# ----------------------------------------------
# MAIN
# ----------------------------------------------

# Load your 3 datasets
datasets = [center_data, left_data, right_data]
sensor_cols = [f"CH_{i}" for i in range(1,9)]

all_X = []
all_y = []

print("\nExtracting features...")
for df in datasets:
    Xf, yf = extract_sliding_features(df, sensor_cols, 15, 1)
    all_X.append(Xf)
    all_y.extend(yf)

final_X = pd.concat(all_X, ignore_index=True)
label_encoder = LabelEncoder()
final_y = label_encoder.fit_transform(all_y)

print("\nSelecting BEST 20 features...")
selector = SelectKBest(score_func=f_classif, k=20)
X20 = selector.fit_transform(final_X, final_y)
best_cols = final_X.columns[selector.get_support()]

print("\nTop 20 features:", best_cols.tolist())
# Save dataset + selected columns
os.makedirs("processed_20", exist_ok=True)

np.save("processed_20/X_20features.npy", X20)
np.save("processed_20/y.npy", final_y)

with open("processed_20/selected_features.pkl", "wb") as f:
    pickle.dump(best_cols.tolist(), f)

print("\nSaved:")
print("  processed_20/X_20features.npy")
print("  processed_20/y.npy")
print("  processed_20/selected_features.pkl")

In [ ]:
for i, cls in enumerate(label_encoder.classes_):
    print(i, cls)

In [ ]:
final_X

In [ ]:
X_selected = np.load("processed_20/X_20features.npy")
y = np.load("processed_20/y.npy")

with open("processed_20/selected_features.pkl", "rb") as f:
    cols = pickle.load(f)

print("X_selected shape:", X_selected.shape)
print("y shape:", y.shape)
print("Selected columns:", cols)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

corr = pd.DataFrame(X_selected, columns=cols).corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Feature Correlation")
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np

# ----------- USE AS 20 FEATURES SELECIONADAS -----------
X = X_selected              # agora é (N amostras, 20 features)
y = final_y                 # rótulos
# -------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

average=np.array([0])
preci=np.array([0])
senci=np.array([0])
bst_est=[]

kf = KFold(n_splits=5, shuffle=True, random_state=7)

def training_and_testing(estimator, param_grid, X_train, y_train, X_test, y_test):
    grid_search = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        cv=kf,
        scoring='accuracy',
        verbose=2
    )

    # Perform the grid search
    grid_search.fit(X_train, y_train)

    # Best parameters and best score
    print("Best parameters:", grid_search.best_params_)
    print("Best cross-validation score: {:.2f}".format(grid_search.best_score_))

    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    sensitivity = recall_score(y_test, y_pred, average='weighted')

    print(f"Final Test Accuracy: {accuracy:.2f}")
    print(f"Final Test Precision: {precision:.2f}")
    print(f"Final Test Sensitivity (Recall): {sensitivity:.2f}")

    # Confusion matrix
    print("\nFinal Confusion Matrix (on test data):")
    mat_con = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=mat_con)
    disp.plot()
    plt.show()

    return accuracy, precision, sensitivity, grid_search


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
import joblib
# Define the model and parameters to tune
knn = KNeighborsClassifier()
param_grid = {
    'n_neighbors': [2, 4, 6, 8],  # You can extend this list based on your dataset
    #'weights': ['uniform', 'distance'],
    'metric': ['minkowski', 'chebyshev', 'euclidean', 'manhattan']
}

accuracy, precision, sensitivity, grid_search = training_and_testing(knn, param_grid, X_train, y_train, X_test, y_test)
print("Accuracy: ", accuracy)
print("Precision: ", precision)
print("Sensitivity: ", sensitivity)
average = np.vstack((average,accuracy))
preci=np.vstack((preci,precision))
senci=np.vstack((senci,sensitivity))
bst_est.append(grid_search.best_params_)

# --- SAVE BEST KNN MODEL ---
best_knn = grid_search.best_estimator_
joblib.dump(best_knn, "db/knn_model_20_features.joblib")
print("✔ Saved best KNN model → db/knn_model_20_features.joblib")

In [ ]:
from sklearn.naive_bayes import GaussianNB
import joblib
# Define the model and parameters to tune
gnb = GaussianNB()
param_grid = {
    'var_smoothing': np.logspace(0,-9, num=100)
}

accuracy, precision, sensitivity, grid_search = training_and_testing(gnb, param_grid, X_train, y_train, X_test, y_test)
print("Accuracy: ", accuracy)
print("Precision: ", precision)
print("Sensitivity: ", sensitivity)
average = np.vstack((average,accuracy))
preci=np.vstack((preci,precision))
senci=np.vstack((senci,sensitivity))
bst_est.append(grid_search.best_params_)

# --- SAVE BEST GAUSSIANNB MODEL ---
best_gnb = grid_search.best_estimator_
joblib.dump(best_gnb, "db/gnb_model_20_features.joblib")
print("✔ Saved best GaussianNB model → db/gnb_model_20_features.joblib")

In [ ]:
# from sklearn.linear_model import LogisticRegression
# # Define the model and parameters to tune
# log_reg = LogisticRegression(max_iter=10000)  # Increasing max_iter for convergence with larger datasets or more complex models
# param_grid = {
#     'C': np.logspace(-4, 4, 20),  # Explore a range of regularization strengths
#     'penalty': ['l1', 'l2']#,  # 'l1', 'l2', 'elasticnet' might be options, depending on the solver
#     #'solver': ['lbfgs', 'liblinear']  # Suitable solvers for small to medium datasets
# }

# accuracy, precision, sensitivity, grid_search = training_and_testing(log_reg, param_grid, X_train, y_train, X_test, y_test)
# print("Accuracy: ", accuracy)
# print("Precision: ", precision)
# print("Sensitivity: ", sensitivity)
# average = np.vstack((average,accuracy))
# preci=np.vstack((preci,precision))
# senci=np.vstack((senci,sensitivity))
# bst_est.append(grid_search.best_params_) 

In [ ]:
from sklearn.tree import DecisionTreeClassifier
import joblib
# Define the model and parameters to tune
decision_tree = DecisionTreeClassifier()
param_grid = {
    'max_depth': [5, 10, 15, 20, 25, 30],  # None means no limit
    'min_samples_split': [2, 4, 6, 8],
    'min_samples_leaf': [2, 4, 6, 8]
}

accuracy, precision, sensitivity, grid_search = training_and_testing(decision_tree, param_grid, X_train, y_train, X_test, y_test)
print("Accuracy: ", accuracy)
print("Precision: ", precision)
print("Sensitivity: ", sensitivity)
average = np.vstack((average,accuracy))
preci=np.vstack((preci,precision))
senci=np.vstack((senci,sensitivity))
bst_est.append(grid_search.best_params_)

# --- SAVE BEST DECISION TREE MODEL ---
best_dt = grid_search.best_estimator_
joblib.dump(best_dt, "db/decision_tree_20_features_model.joblib")
print("✔ Saved best Decision Tree model → db/decision_tree_20_features_model.joblib")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import joblib
# Define the model and parameters to tune
random_forest = RandomForestClassifier()
param_grid = {
    'n_estimators': [100, 200, 300],  # Number of trees in the forest
    'max_depth': [5, 10, 15, 20],  # Maximum depth of the tree
    'min_samples_split': [2, 4, 6, 8],  # Minimum number of samples required to split an internal node
    'min_samples_leaf': [2, 4, 6, 8]  # Minimum number of samples required at each leaf node
}

accuracy, precision, sensitivity, grid_search = training_and_testing(random_forest, param_grid, X_train, y_train, X_test, y_test)
print("Accuracy: ", accuracy)
print("Precision: ", precision)
print("Sensitivity: ", sensitivity)
average = np.vstack((average,accuracy))
preci=np.vstack((preci,precision))
senci=np.vstack((senci,sensitivity))
bst_est.append(grid_search.best_params_)

# --- SAVE BEST RANDOM FOREST MODEL ---
best_rf = grid_search.best_estimator_
joblib.dump(best_rf, "db/random_forest_model_20_features.joblib")
print("✔ Saved best Random Forest model → db/random_forest_model_20_features.joblib")

In [ ]:
from sklearn.model_selection import RandomizedSearchCV # Assuming you use KFold
def training_and_testing_svm(estimator, param_grid, X_train, y_train, X_test, y_test):
    grid_search = RandomizedSearchCV(estimator=estimator, param_distributions=param_grid, cv=kf, scoring='accuracy', verbose=3, n_iter=50)  # n_iter can be adjusted based on the number of combinations you want to try

    # Perform the grid search
    grid_search.fit(X_train, y_train)

    # Best parameters and best score
    print("Best parameters:", grid_search.best_params_)
    print("Best cross-validation score: {:.2f}".format(grid_search.best_score_))

    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)

    # Calculate and print all final metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    sensitivity = recall_score(y_test, y_pred, average='weighted') # Sensitivity is the same as recall

    print(f"Final Test Accuracy: {accuracy:.2f}")
    print(f"Final Test Precision: {precision:.2f}")
    print(f"Final Test Sensitivity (Recall): {sensitivity:.2f}")

    # Display the final confusion matrix
    print("\nFinal Confusion Matrix (on test data):")
    mat_con = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=mat_con)
    disp.plot()
    plt.show()
    return accuracy, precision, sensitivity, grid_search

from sklearn.svm import SVC
from sklearn.datasets import make_classification # Import make_classification
from sklearn.model_selection import train_test_split # Import train_test_split

from sklearn.svm import SVC
# Define the model and parameters to tune
svm = SVC()
param_grid = [
    {'kernel': ['rbf'], 'C': [0.1, 1, 10, 100], 'gamma': ['scale', 1, 0.1, 0.01]},
    #{'kernel': ['linear'], 'C': [0.1, 1, 10, 100]}
]

accuracy, precision, sensitivity, grid_search = training_and_testing_svm(svm, param_grid, X_train, y_train, X_test, y_test)
print("Accuracy: ", accuracy)
print("Precision: ", precision)
print("Sensitivity: ", sensitivity)
average = np.vstack((average,accuracy))
preci=np.vstack((preci,precision))
senci=np.vstack((senci,sensitivity))
bst_est.append(grid_search.best_params_)

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import joblib

lda = LinearDiscriminantAnalysis()
param_grid = {
    'solver': ['svd', 'lsqr', 'eigen'],
    'shrinkage': [None, 'auto', 0.1, 0.5, 0.9]
}

accuracy, precision, sensitivity, grid_search = training_and_testing(lda, param_grid, X_train, y_train, X_test, y_test)
print("Accuracy: ", accuracy)
print("Precision: ", precision)
print("Sensitivity: ", sensitivity)
average = np.vstack((average,accuracy))
preci=np.vstack((preci,precision))
senci=np.vstack((senci,sensitivity))
bst_est.append(grid_search.best_params_)

# --- SAVE BEST LDA MODEL ---
best_lda = grid_search.best_estimator_
joblib.dump(best_lda, "db/lda_model_20_features.joblib")
print("✔ Saved best LDA model → db/lda_model_20_features.joblib")

In [ ]:
from sklearn.neural_network import MLPClassifier
import joblib

mlp = MLPClassifier(max_iter=1000)  # Increased max_iter for better convergence
param_grid = {'alpha': [0.0001, 0.001, 0.01, 0.1, 1],
              'hidden_layer_sizes': [(50,), (100,), (50, 50), (100, 50)],
              'activation': ['tanh  ', 'relu'],
              'solver': ['adam', 'sgd']}

accuracy, precision, sensitivity, grid_search = training_and_testing(mlp, param_grid, X_train, y_train, X_test, y_test)
print("Accuracy: ", accuracy)
print("Precision: ", precision)
print("Sensitivity: ", sensitivity)
average = np.vstack((average,accuracy))
preci=np.vstack((preci,precision))
senci=np.vstack((senci,sensitivity))
bst_est.append(grid_search.best_params_)

# --- SAVE BEST MLP MODEL ---
best_mlp = grid_search.best_estimator_
joblib.dump(best_mlp, "db/mlp_model_20_features.joblib")
print("✔ Saved best MLP model → db/mlp_model_20_features.joblib")

In [ ]:
# prompt: take  average preci and sensci  and consider this, rows are porcentages so must be *100 and it must be first column value +/- second column value, storage them as csv
import pandas as pd
def create_df(average, preci, senci, bst_est):
    # Convert numpy arrays to DataFrames
    average_df = pd.DataFrame(average, columns=['Accuracy'])
    preci_df = pd.DataFrame(preci, columns=['Precision'])
    senci_df = pd.DataFrame(senci, columns=['Sensitivity'])

    # Multiply the columns by 100 for percentage
    average_df[['Accuracy']] *= 100
    preci_df[['Precision']] *= 100
    senci_df[['Sensitivity']] *= 100

    # Combine mean and std columns into a single string format for each metric
    average_df['Accuracy'] = average_df['Accuracy'].round(2).astype(str) 
    preci_df['Precision'] = preci_df['Precision'].round(2).astype(str)
    senci_df['Sensitivity'] = senci_df['Sensitivity'].round(2).astype(str)

    results_df = pd.DataFrame({
    'Model': ['KNN', 'GaussianNB', 'Decision Tree', 'Random Forest', 'SVM', 'LDA', 'MLP'],
    'Accuracy': average_df['Accuracy'].iloc[1:].values,
    'Precision': preci_df['Precision'].iloc[1:].values,
    'Sensitivity': senci_df['Sensitivity'].iloc[1:].values,
    'Best Estimator Params': bst_est
    })
    return results_df

results_df = create_df(average, preci, senci, bst_est)
# Save the results to a CSV file
results_df.to_csv('model_performance_summary_force_hybrid.csv', index=False)

print("\nModel Performance Summary:")
results_df